# From checkpoint to endpoint: merge, compress, serve

LLM Training Lab, module 22. Regime POST-HOC: no new fine-tuning. You merge two of your earlier fine-tunes of Qwen3-0.6B-Base, compress the merge three ways, and serve it.

Steps 1 to 3 and 6 run on a CPU. Steps 4, 5, 8 and 9 need a GPU (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2, Internet on). Steps 7 and 11 run llama.cpp on the CPU of the same session. Code cell numbers match the lab page.

**Free T4 path: `FROM_OUR_CHECKPOINT = True`, set near the top of code cell 1, and the default.** The long steps then use our recorded results from the Hugging Face Hub instead of running again: step 5 scores our merge, step 7 loads our GGUF file, step 9 prints our QAT run and step 12 prints our vLLM run. Every later step still runs on them. Set it to `False` to run every step yourself, which took 75.5 min on our T4 VM against 34.0 min. Steps 5 and 12 have two more switches, `RUN_GRID` and `RUN_VLLM`.

<!-- llt-run-help -->
## How to run this notebook

- **Check the lab card** on the module page first. It says whether this lab needs a GPU and how long it took us.
- **Kaggle** (where we recorded most runs): use the lab page's *Open on Kaggle* button. If the lab needs a GPU, open the notebook settings and pick *Accelerator: GPU T4 x2* (not P100). Switch *Internet* on so the install cell and the downloads work. Kaggle asks you to verify a phone number once before it allows a GPU or internet.
- **Colab**: use the lab page's *Open in Colab* button. If the lab needs a GPU, pick *Runtime > Change runtime type > T4 GPU*.
- **Hugging Face**: no account or token is needed. Our checkpoints and data files download from the public repo [aimlcompanion/llm-training-lab](https://huggingface.co/aimlcompanion/llm-training-lab).
- **Run the cells in order.** Code cell numbers match the lab page. Your numbers should land close to ours, not match exactly: GPU runs are not bit-for-bit repeatable.

In [ ]:
import os, sys, platform, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one GPU, set before torch loads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
problems = []
if sys.version_info < (3, 11):
    problems.append("Python 3.11 or newer is needed (3.12 recommended). Start a current Colab or Kaggle session.")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:                                   # a CPU runtime has no nvidia-smi
    smi = []
gpu = smi[0] if smi else ""
if not gpu:
    problems.append("No GPU: steps 1 to 3 and 6 still run. For steps 4, 5, 8 and 9, Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    problems.append("A P100 has no kernels in current PyTorch. Pick a T4.")
cache = os.path.expanduser("~/.cache")
os.makedirs(cache, exist_ok=True)
free_disk = shutil.disk_usage(cache).free / 1e9
if free_disk < 20:
    problems.append("Free at least 20 GB of disk for the model and data downloads (" + str(round(free_disk)) + " GB free).")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = "not installed"
print("Python", platform.python_version(), "| GPU", gpu or "none", "| torch", torch_version, "| disk free", round(free_disk), "GB")
for p in problems:
    print("PROBLEM", p)
print("Runtime OK" if not problems else "Read the PROBLEM lines first.")

In [ ]:
# Pinned stack of our recorded run. torch and numpy stay as the runtime ships them.
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 tokenizers==0.23.3 huggingface_hub==2.4.0 mergekit==0.1.5 gguf==0.19.0

## Step 1. Check that your fine-tunes share a base (CPU)

Code cell 1 on the lab page.

In [ ]:
# The long steps have an exit. True (the default, built for a free T4): steps 5, 7, 9 and 12 use our recorded results from the Hub
# instead of running again, and every later step still runs on them. False: you run every step yourself, as we did.
FROM_OUR_CHECKPOINT = True
import json, os, struct
import torch
from huggingface_hub import HfFileSystem, hf_hub_download

torch.set_num_threads(4)
HUB = "aimlcompanion/llm-training-lab"
fs = HfFileSystem()
REC = json.load(open(hf_hub_download(HUB, "lltShip/recorded.json")))     # our A100 results for the long steps, 5 KB

def adapter_base(path):
    # The base model an adapter was trained on, from its adapter_config.json on the Hub
    return json.loads(fs.read_text(HUB + "/" + path + "/adapter_config.json"))["base_model_name_or_path"]

def tensor_shapes(path):
    # Read only the header of a .safetensors file on the Hub: names and shapes, no weights downloaded
    with fs.open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
    return {k: v["shape"] for k, v in head.items() if k != "__metadata__"}

print("lltToolUse adapter was trained on", adapter_base("lltToolUse/qwen3.5-2b-lora-s0"))
print("lltDistillation adapter was trained on", adapter_base("lltDistillation/onpolicy-reverse-s0"))
for path in ["lltDPO/sft-s0", "lltDomainAdaptation/cpt-r5-lr3e-5"]:
    cfg = json.loads(fs.read_text(HUB + "/" + path + "/config.json"))
    print(path, "is a full model |", cfg["architectures"][0], "| hidden", cfg["hidden_size"], "| layers", cfg["num_hidden_layers"],
          "| vocab", cfg["vocab_size"])

tool = tensor_shapes(HUB + "/lltToolUse/qwen3.5-2b-lora-s0/adapter_model.safetensors")
qa = [k for k in tool if "q_proj.lora_A" in k][0]
ours = tensor_shapes("Qwen/Qwen3-0.6B-Base/model.safetensors")
print("ToolUse LoRA A for q_proj", tool[qa], "expects inputs of width", tool[qa][1])
print("Qwen3-0.6B-Base q_proj weight", ours["model.layers.0.self_attn.q_proj.weight"], "has inputs of width",
      ours["model.layers.0.self_attn.q_proj.weight"][1])

# Same shapes are not enough. Train a tiny network, then permute its hidden units: the function does not change,
# but the weights now sit in different places, so averaging the two copies breaks it.
g = torch.Generator().manual_seed(0)
x = torch.linspace(-2, 2, 256)[:, None]
y = torch.sin(3 * x) + 0.5 * x

def mlp(w, x):
    return torch.tanh(x @ w["W1"] + w["b1"]) @ w["W2"] + w["b2"]

def loss(w):
    return float(((mlp(w, x) - y) ** 2).mean())

def train(w, steps, lr, seed):
    w = {k: v.clone().requires_grad_(True) for k, v in w.items()}
    opt = torch.optim.Adam(w.values(), lr=lr)
    order = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        idx = torch.randint(0, 256, (64,), generator=order)
        l = ((mlp(w, x[idx]) - y[idx]) ** 2).mean()
        opt.zero_grad(); l.backward(); opt.step()
    return {k: v.detach() for k, v in w.items()}

init = {"W1": torch.randn(1, 64, generator=g), "b1": torch.randn(64, generator=g) * 0.1,
        "W2": torch.randn(64, 1, generator=g) * 0.1, "b2": torch.zeros(1)}
A = train(init, 3000, 1e-2, seed=1)                       # the "base"
perm = torch.randperm(64, generator=g)
B = {"W1": A["W1"][:, perm], "b1": A["b1"][perm], "W2": A["W2"][perm], "b2": A["b2"]}   # same function, units shuffled
C = train(A, 300, 1e-3, seed=2)                           # a short fine-tune that starts from A
avg = lambda P, Q: {k: (P[k] + Q[k]) / 2 for k in P}
print("loss A", round(loss(A), 5), "| permuted copy B", round(loss(B), 5), "| max output gap A vs B",
      float((mlp(A, x) - mlp(B, x)).abs().max()))
print("average of A and B", round(loss(avg(A, B)), 5), "| average of A and its fine-tune C", round(loss(avg(A, C)), 5))

## Step 2. Compute the task vectors and look at them (CPU)

Code cell 2 on the lab page.

In [ ]:
import glob, time
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

BASE = "Qwen/Qwen3-0.6B-Base"
hub_dir = snapshot_download(HUB, allow_patterns=["lltDPO/sft-s0/*", "lltDomainAdaptation/cpt-r5-lr3e-5/*"])
DIRS = {"base": snapshot_download(BASE), "sft": hub_dir + "/lltDPO/sft-s0", "hindi": hub_dir + "/lltDomainAdaptation/cpt-r5-lr3e-5"}

def load_weights(d):
    # Every tensor of a checkpoint folder, in the dtype it was saved in (bf16 for the base, fp16 for our fine-tunes)
    sd = {}
    for f in sorted(glob.glob(d + "/*.safetensors")):
        sd.update(load_file(f))
    return sd

W = {name: load_weights(d) for name, d in DIRS.items()}
print({name: (len(sd), str(next(iter(sd.values())).dtype)) for name, sd in W.items()})
assert W["base"].keys() == W["sft"].keys() == W["hindi"].keys(), "different tensor names: not the same architecture"

def tau(name, k):
    # Task vector of one tensor: fine-tuned weights minus base weights, in fp32
    return W[name][k].float() - W["base"][k].float()

def group(k):
    if "embed_tokens" in k:
        return "embeddings"
    return "norms" if "norm" in k else ("attention" if "self_attn" in k else "mlp")

stats = {n: {"sq": {}, "nonzero": 0} for n in ("sft", "hindi")}
dot = conflict = both = total = base_sq = 0.0
t0 = time.time()
for k in W["base"]:
    a, b = tau("sft", k), tau("hindi", k)
    for n, d in (("sft", a), ("hindi", b)):
        stats[n]["sq"][group(k)] = stats[n]["sq"].get(group(k), 0.0) + float((d * d).sum())
        stats[n]["nonzero"] += int((d != 0).sum())
    base_sq += float((W["base"][k].float() ** 2).sum())
    dot += float((a * b).sum())
    nz = (a != 0) & (b != 0)
    both += int(nz.sum())
    conflict += int((nz & (a.sign() != b.sign())).sum())
    total += a.numel()

norm = {n: sum(s["sq"].values()) ** 0.5 for n, s in stats.items()}
for n, s in stats.items():
    print(n.ljust(6), "| norm", round(norm[n], 2), "=", round(100 * norm[n] / base_sq ** 0.5, 3), "% of the base norm",
          "| nonzero entries", round(100 * s["nonzero"] / total, 1), "%",
          "| share of the squared norm", {g: round(v / norm[n] ** 2, 3) for g, v in s["sq"].items()})
print("cosine(sft, hindi)", round(dot / (norm["sft"] * norm["hindi"]), 4), "| entries both changed", round(100 * both / total, 1),
      "% | of those, opposite signs", round(100 * conflict / both, 1), "% |", round(time.time() - t0), "s")

k = "model.layers.13.self_attn.q_proj.weight"
for n in ("sft", "hindi"):
    s = torch.linalg.svdvals(tau(n, k))
    print(n.ljust(6), k, "singular values 1, 16, 17 relative to the first:", [round(float(s[i] / s[0]), 4) for i in (0, 15, 16)])

## Step 3. Write linear, task arithmetic, TIES and DARE (CPU)

Code cell 3 on the lab page.

In [ ]:
def task_arithmetic(base, deltas, lams):
    # base + lambda_1 tau_1 + lambda_2 tau_2 + ...
    out = base.float().clone()
    for d, lam in zip(deltas, lams):
        out += lam * d
    return out

def linear(models, weights):
    # A weighted average of whole models, a "model soup". With a shared base it is task arithmetic with lambda_i = w_i / sum(w)
    return sum(w * m.float() for m, w in zip(models, weights)) / sum(weights)

def trim(d, density):
    # TIES step 1: keep the largest-magnitude share density of the entries, zero the rest
    if density >= 1:
        return d
    k = max(1, int(density * d.numel()))
    keep = torch.argsort(d.abs().reshape(-1), descending=True)[:k]
    mask = torch.zeros(d.numel(), dtype=d.dtype)
    mask[keep] = 1
    return d * mask.reshape(d.shape)

def ties(base, deltas, density, lam=1.0, weights=None):
    weights = weights or [1.0] * len(deltas)
    w = torch.tensor(weights).reshape(-1, *[1] * base.dim())
    t = torch.stack([trim(d, density) for d in deltas]) * w      # trimmed, weighted task vectors
    sign = torch.where(t.sum(0) >= 0, 1.0, -1.0)                  # step 2: elect the sign with the larger total mass
    agree = t.sign() == sign                                      # entries that agree; a trimmed zero never agrees
    count = (w * agree).sum(0)
    merged = (t * agree).sum(0) / torch.where(count == 0, 1.0, count)   # step 3: disjoint mean over the agreeing ones
    return base.float() + lam * merged

def dare(d, p, gen):
    # DARE: drop each entry with probability p, scale the survivors by 1 / (1 - p) so the expected value is unchanged
    keep = torch.bernoulli(torch.full(d.shape, 1 - p), generator=gen)
    return d * keep / (1 - p)

# Check every function on four numbers you can follow by hand
b = torch.zeros(4)
t1, t2 = torch.tensor([0.5, -0.2, 0.1, 0.0]), torch.tensor([0.3, 0.4, -0.6, 0.05])
r4 = lambda v: [round(x, 4) for x in v.tolist()]
print("task arithmetic, lambda 1 ", r4(task_arithmetic(b, [t1, t2], [1, 1])))
print("linear, equal weights     ", r4(linear([b + t1, b + t2], [1, 1])))
print("trim to density 0.5       ", r4(trim(t1, 0.5)), r4(trim(t2, 0.5)))
print("TIES, density 0.5         ", r4(ties(b, [t1, t2], 0.5)))
big = torch.full((100000,), 0.01)
dropped = dare(big, 0.9, torch.Generator().manual_seed(0))
print("DARE p 0.9 keeps", int((dropped != 0).sum()), "of 100000 | mean before", round(float(big.mean()), 5),
      "after", round(float(dropped.mean()), 5))
assert torch.allclose(task_arithmetic(b, [t1, t2], [0.5, 0.5]), linear([b + t1, b + t2], [1, 1]))
assert torch.allclose(ties(b, [t1, t2], 0.5), torch.tensor([0.5, 0.4, -0.6, 0.0]))

# One real tensor: what TIES keeps from each fine-tune
k = "model.layers.13.mlp.down_proj.weight"
d = [tau("sft", k), tau("hindi", k)]
t = torch.stack([trim(x, 0.2) for x in d])
sign = torch.where(t.sum(0) >= 0, 1.0, -1.0)
for i, n in enumerate(("sft", "hindi")):
    kept, agree = t[i] != 0, t[i].sign() == sign
    print(n.ljust(6), "| kept by trim", round(100 * float(kept.float().mean()), 1), "% | of those, lost the sign vote",
          round(100 * float((kept & ~agree).sum() / kept.sum()), 1), "% | share of its squared norm kept after the vote",
          round(float((t[i] * agree).pow(2).sum() / d[i].pow(2).sum()), 3))

def merge_model(method, lams=(1.0, 1.0), density=1.0, p=0.0, seed=0, dtype=torch.float16):
    # Merge every tensor of the two fine-tunes into the base, one tensor at a time, on the CPU
    gen = torch.Generator().manual_seed(seed)
    out = {}
    for k in W["base"]:
        base, deltas = W["base"][k], [tau("sft", k), tau("hindi", k)]
        if method == "ties":
            m = ties(base, deltas, density, lam=lams[0])
        elif method == "dare":
            m = task_arithmetic(base, [dare(x, p, gen) for x in deltas], lams)
        else:
            m = task_arithmetic(base, deltas, lams)
        out[k] = m.to(dtype)
    return out

## Step 4. Build a two-skill panel and score the parents (GPU)

Code cell 4 on the lab page.

In [ ]:
import gc, math
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_TEXT, N_REPLY, N_GEN = (200, 300, 100) if DEV.type == "cuda" else (20, 30, 0)   # a CPU scores a small slice, no generation
tok = AutoTokenizer.from_pretrained(BASE)
NL = chr(10)
TEMPLATE = "### Instruction:" + NL + "{}" + NL + NL + "### Response:" + NL     # the plain-text template of the lltDPO SFT model
runs = {}                                                                      # every number this notebook prints is kept here

def cut(text, n_chars):
    # The first n_chars characters, cut back to the last space (the lltDomainAdaptation panel recipe)
    return text[:n_chars].rsplit(" ", 1)[0] if len(text) > n_chars else None

def held_texts(name, config, split, n_docs, n_chars, want):
    out = []
    for r in load_dataset(name, name=config, split=split, streaming=True).take(n_docs):
        t = cut(r["text"], n_chars)
        if t and len(tok(t).input_ids) <= 1023:
            out.append(t)
        if len(out) == want:
            break
    return out

hindi = held_texts("HuggingFaceFW/fineweb-2", "hin_Deva", "test", 400, 1000, N_TEXT)
english = held_texts("HuggingFaceFW/fineweb-edu", "sample-10BT", "train", 500, 3000, N_TEXT)
replies, prompts = [], []
for ex in load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs"):
    p = tok(TEMPLATE.format(ex["prompt"])).input_ids
    c, r = ex["chosen"][-1]["content"].strip(), ex["rejected"][-1]["content"].strip()
    full = tok(TEMPLATE.format(ex["prompt"]) + c).input_ids + [tok.eos_token_id]
    if c != r and len(full) <= 512 and len(tok(TEMPLATE.format(ex["prompt"]) + r).input_ids) + 1 <= 512:
        replies.append((full, len(p)))
        if len(p) <= 200:
            prompts.append(ex["prompt"])
    if len(replies) == N_REPLY:
        break
prompts = prompts[:N_GEN]
print(len(hindi), "Hindi texts |", len(english), "English texts |", len(replies), "held-out replies |", len(prompts), "prompts")

@torch.no_grad()
def nll_items(model, rows, starts, bs):
    # Summed NLL in nats and token count of each row, scoring the tokens from position starts[i] on
    order = sorted(range(len(rows)), key=lambda i: -len(rows[i]))
    nll, cnt = np.zeros(len(rows)), np.zeros(len(rows))
    for j in range(0, len(order), bs):
        idx = order[j:j + bs]
        L = max(len(rows[i]) for i in idx)
        ids = torch.full((len(idx), L), tok.eos_token_id)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for a, i in enumerate(idx):
            ids[a, :len(rows[i])] = torch.tensor(rows[i])
            att[a, :len(rows[i])] = 1
        logits = model(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits[:, :-1].float()
        lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None].to(DEV))[..., 0]
        for a, i in enumerate(idx):
            nll[i] = -float(lp[a, starts[i] - 1:len(rows[i]) - 1].sum())
            cnt[i] = len(rows[i]) - starts[i]
    return nll, cnt

@torch.no_grad()
def end_rate(model, max_new=256, bs=25):
    # Greedy replies to held-out prompts: does the model stop on its own with <|endoftext|>?
    tok.padding_side = "left"
    ended, lengths, texts = [], [], []
    for j in range(0, len(prompts), bs):
        enc = tok([TEMPLATE.format(p) for p in prompts[j:j + bs]], return_tensors="pt", padding=True).to(DEV)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for row in out[:, enc.input_ids.shape[1]:].tolist():
            stop = tok.eos_token_id in row
            n = row.index(tok.eos_token_id) + 1 if stop else len(row)
            ended.append(float(stop)); lengths.append(n); texts.append(tok.decode(row[:n], skip_special_tokens=True))
    return np.array(ended), np.array(lengths), texts

H_ROWS = [[tok.eos_token_id] + tok(t).input_ids for t in hindi]
E_ROWS = [[tok.eos_token_id] + tok(t).input_ids for t in english]
H_BYTES = np.array([len(t.encode("utf-8")) for t in hindi])
E_BYTES = np.array([len(t.encode("utf-8")) for t in english])

def panel(model):
    # The two skills and one forgetting check, with per-item numbers kept for paired intervals
    h, _ = nll_items(model, H_ROWS, [1] * len(H_ROWS), bs=2)
    e, _ = nll_items(model, E_ROWS, [1] * len(E_ROWS), bs=2)
    r, n = nll_items(model, [x[0] for x in replies], [x[1] for x in replies], bs=4)
    out = {"hiBpb": h.sum() / math.log(2) / H_BYTES.sum(), "enBpb": e.sum() / math.log(2) / E_BYTES.sum(),
           "replyNll": r.sum() / n.sum(), "items": {"hi": h, "en": e, "reply": r, "replyTok": n}}
    if prompts:
        ended, lengths, texts = end_rate(model)
        out.update(endRate=ended.mean(), meanTokens=lengths.mean(), items=dict(out["items"], ended=ended), sample=texts[0])
    return out

def ci(a, b, key, n=10000, seed=0):
    # 95% paired bootstrap interval of (b minus a) over the same items
    num = {"hi": ("hi", H_BYTES), "en": ("en", E_BYTES), "reply": ("reply", "replyTok"), "ended": ("ended", None)}[key]
    rng = np.random.default_rng(seed)
    N = len(a["items"][num[0]])
    i = rng.integers(0, N, (n, N))
    def stat(m):
        top = m["items"][num[0]][i].sum(1)
        if num[1] is None:
            return top / N
        den = num[1][i].sum(1) * math.log(2) if isinstance(num[1], np.ndarray) else m["items"][num[1]][i].sum(1)
        return top / den
    d = stat(b) - stat(a)
    return [round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4)]

model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16 if DEV.type == "cuda" else torch.float32).to(DEV).eval()

def load_into(sd):
    # Put a state dict into the one model on the GPU; lm_head is tied to the embeddings, so it is not in the file
    missing, unexpected = model.load_state_dict({k: v.to(model.dtype) for k, v in sd.items()}, strict=False)
    assert set(missing) <= {"lm_head.weight"} and not unexpected, (missing, unexpected)

def show(name, m, ref=None):
    line = name.ljust(22) + " | Hindi bpb " + format(m["hiBpb"], ".4f") + " | English bpb " + format(m["enBpb"], ".4f")
    line += " | reply NLL " + format(m["replyNll"], ".4f")
    if "endRate" in m:
        line += " | ends " + format(m["endRate"], ".2f") + ", " + format(m["meanTokens"], ".0f") + " tokens"
    print(line)

t0 = time.time()
for name in ("base", "sft", "hindi"):
    load_into(W[name])
    runs[name] = panel(model)
    show(name, runs[name])
print("Hindi bpb, hindi minus base", ci(runs["base"], runs["hindi"], "hi"), "| reply NLL, sft minus base",
      ci(runs["base"], runs["sft"], "reply"), "|", round(time.time() - t0), "s")

## Step 5. Merge, score both skills, pick one (FROM_OUR_CHECKPOINT scores our merge, RUN_GRID = True scores 9 more)

Code cell 5 on the lab page.

In [ ]:
RUN_GRID = False      # True scores 9 more merges (each panel took about 1 min on our A100 and 2 min on our T4). With FROM_OUR_CHECKPOINT on, only our best merge is scored

def kept(m):
    # Share of each fine-tune's gain the merge keeps: 1.0 = as good as that fine-tune, 0 = no better than the base
    hi = (runs["base"]["hiBpb"] - m["hiBpb"]) / (runs["base"]["hiBpb"] - runs["hindi"]["hiBpb"])
    rep = (runs["base"]["replyNll"] - m["replyNll"]) / (runs["base"]["replyNll"] - runs["sft"]["replyNll"])
    return round(float(hi), 3), round(float(rep), 3)

GRID = [("average, lambda 0.5", "ta", dict(lams=(0.5, 0.5))),
        ("task arithmetic 1.0", "ta", dict(lams=(1.0, 1.0))),
        ("TIES density 0.2", "ties", dict(density=0.2))]
if RUN_GRID:
    GRID += [("task arithmetic 0.75", "ta", dict(lams=(0.75, 0.75))),
             ("task arithmetic 1.25", "ta", dict(lams=(1.25, 1.25))),
             ("task arithmetic 1.0 / 0.5", "ta", dict(lams=(1.0, 0.5))),       # (lambda for SFT, lambda for Hindi)
             ("task arithmetic 1.0 / 0.25", "ta", dict(lams=(1.0, 0.25))),
             ("TIES density 0.5", "ties", dict(density=0.5)),
             ("TIES density 1.0", "ties", dict(density=1.0)),
             ("DARE p 0.5", "dare", dict(p=0.5, seed=0)),
             ("DARE p 0.9", "dare", dict(p=0.9, seed=0)),
             ("DARE p 0.9 seed 1", "dare", dict(p=0.9, seed=1))]
elif FROM_OUR_CHECKPOINT:
    # Score only the merge we shipped, and read the other eight from our grid, so the rule and every later step match the page
    GRID.append(("task arithmetic 1.0 / 0.5", "ta", dict(lams=(1.0, 0.5))))
    for n, m in REC["grid"].items():
        if n not in [g[0] for g in GRID]:
            print((n.ljust(26) + " | Hindi bpb " + format(m["hiBpb"], ".4f") + " | reply NLL " + format(m["replyNll"], ".4f")
                   + " | kept " + str(m["kept"])), "| recorded on our A100")
for name, method, kw in GRID:
    t0 = time.time()
    sd = merge_model(method, **kw)
    t1 = time.time()
    load_into(sd)
    runs[name] = panel(model)
    runs[name]["kept"] = kept(runs[name])
    runs[name]["mergeSec"], runs[name]["panelSec"] = round(t1 - t0, 1), round(time.time() - t1, 1)
    show(name, runs[name])
    print("   kept (Hindi, replies)", runs[name]["kept"], "| merge", runs[name]["mergeSec"], "s on the CPU, panel",
          runs[name]["panelSec"], "s")

# Pick the merge whose weaker skill keeps the most of its gain, then save it for parts 2 and 3
best = max((n for n, _, _ in GRID), key=lambda n: min(runs[n]["kept"]))
print("ship:", best, "| Hindi bpb vs hindi model", ci(runs["hindi"], runs[best], "hi"),
      "| reply NLL vs sft model", ci(runs["sft"], runs[best], "reply"), "| English bpb vs base", ci(runs["base"], runs[best], "en"))
method, kw = {n: (m, k) for n, m, k in GRID}[best]
load_into(merge_model(method, **kw))
model.save_pretrained("merged")
tok.chat_template = ("{% for m in messages %}{% if m['role'] == 'user' %}### Instruction:" + NL + "{{ m['content'] }}" + NL + NL
                     + "### Response:" + NL + "{% elif m['role'] == 'assistant' %}{{ m['content'] }}<|endoftext|>{% endif %}{% endfor %}")
tok.save_pretrained("merged")
print(sorted(os.listdir("merged")))

## Step 6. Reproduce your TIES merge with mergekit (CPU)

Code cell 6 on the lab page.

In [ ]:
import shutil, subprocess, sys
from safetensors import safe_open

CFG = f"""merge_method: ties
base_model: {DIRS["base"]}
models:
  - model: {DIRS["sft"]}
    parameters: {{weight: 1.0, density: 0.2}}
  - model: {DIRS["hindi"]}
    parameters: {{weight: 1.0, density: 0.2}}
parameters: {{normalize: true, lambda: 1.0}}
dtype: float32
out_dtype: float32
"""
open("ties.yml", "w").write(CFG)
t0 = time.time()
subprocess.run(["mergekit-yaml", "ties.yml", "merged-mergekit", "--quiet", "--random-seed", "0"], check=True)
print("mergekit took", round(time.time() - t0), "s")

t0 = time.time()
ours = merge_model("ties", density=0.2, dtype=torch.float32)
print("your TIES took", round(time.time() - t0), "s")
worst, same, n = 0.0, 0, 0
for f in sorted(glob.glob("merged-mergekit/*.safetensors")):
    with safe_open(f, "pt") as g:
        for k in g.keys():
            gap = float((g.get_tensor(k) - ours[k]).abs().max())
            worst, same, n = max(worst, gap), same + (gap == 0), n + 1
print(n, "tensors compared |", same, "identical | largest gap", worst)
del ours
shutil.rmtree("merged-mergekit")          # 2.4 GB of fp32 weights you no longer need

## Step 7. Convert to GGUF and compare k-quants (llama.cpp, CPU; FROM_OUR_CHECKPOINT loads our files)

Code cell 7 on the lab page.

In [ ]:
import shutil, subprocess, sys, tarfile, urllib.request
from huggingface_hub import hf_hub_download
from gguf import GGUFReader
from gguf.quants import dequantize

TAG = "b12182"                                   # llama.cpp release tag; quote the tag you used
REL = "https://github.com/ggml-org/llama.cpp/releases/download/" + TAG + "/"
BIN = os.path.abspath("llama-" + TAG)            # prebuilt CPU binaries for Linux x64
SRC = os.path.abspath("llama.cpp-" + TAG)        # the same tag's source, for convert_hf_to_gguf.py and gguf-py
if not os.path.isdir(BIN):
    urllib.request.urlretrieve(REL + "llama-" + TAG + "-bin-ubuntu-x64.tar.gz", "bin.tgz")
    tarfile.open("bin.tgz").extractall(".", filter="data")
if not FROM_OUR_CHECKPOINT and not os.path.isdir(SRC):
    urllib.request.urlretrieve("https://github.com/ggml-org/llama.cpp/archive/refs/tags/" + TAG + ".tar.gz", "src.tgz")
    tarfile.open("src.tgz").extractall(".", filter="data")

def sh(cmd, **kw):
    # Run a command, return seconds taken; raise with its last output lines if it fails
    t = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if p.returncode:
        raise RuntimeError(p.stdout[-2000:] + p.stderr[-2000:])
    return round(time.time() - t, 1), p.stdout + p.stderr

merged_sd = {k: v.float().cpu() for k, v in model.state_dict().items() if k != "lm_head.weight"}   # the merge in fp32, kept for steps 8 and 9
if FROM_OUR_CHECKPOINT:
    # Converting to f16, quantizing six ways, scoring five files and measuring perplexity took 11.3 min on our A100 VM.
    # Take our Q4_K_M file of the same merge (step 11 serves it) and read the rest from our recorded run.
    shutil.copyfile(hf_hub_download(HUB, "lltShip/merged-Q4_K_M.gguf"), "merged-Q4_K_M.gguf")
    gguf = REC["gguf"]["files"]
    for q, g in gguf.items():
        print(q.ljust(7), g["MiB"], "MiB | made in", g["sec"], "s | recorded on our A100 VM" + (" | llama-perplexity on the Hindi texts: " + g["ppl"] if "ppl" in g else ""))
    for q, m in REC["gguf"]["panels"].items():
        print(("GGUF " + q).ljust(22), "| Hindi bpb", format(m["hiBpb"], ".4f"), "| English bpb", format(m["enBpb"], ".4f"),
              "| reply NLL", format(m["replyNll"], ".4f"), "| ends", format(m["endRate"], ".2f"), "| tensor types", m["kinds"])
    runs["gguf"] = gguf
else:
    env = dict(os.environ, PYTHONPATH=SRC + "/gguf-py")
    sec, _ = sh([sys.executable, SRC + "/convert_hf_to_gguf.py", "merged", "--outtype", "f16", "--outfile", "merged-f16.gguf"], env=env)
    gguf = {"f16": {"file": "merged-f16.gguf", "sec": sec}}
    for q in ["Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K"]:
        sec, _ = sh([BIN + "/llama-quantize", "merged-f16.gguf", "merged-" + q + ".gguf", q])
        gguf[q] = {"file": "merged-" + q + ".gguf", "sec": sec}
    for q, g in gguf.items():
        g["MiB"] = round(os.path.getsize(g["file"]) / 2 ** 20, 1)
        print(q.ljust(7), g["MiB"], "MiB | made in", g["sec"], "s")

    # Read the quantized weights back into the PyTorch model, so each GGUF file gets the same paired panel as the merges
    NAMES = {"attn_q": "self_attn.q_proj", "attn_k": "self_attn.k_proj", "attn_v": "self_attn.v_proj", "attn_output": "self_attn.o_proj",
             "attn_q_norm": "self_attn.q_norm", "attn_k_norm": "self_attn.k_norm", "attn_norm": "input_layernorm",
             "ffn_norm": "post_attention_layernorm", "ffn_gate": "mlp.gate_proj", "ffn_up": "mlp.up_proj", "ffn_down": "mlp.down_proj"}

    def gguf_weights(path):
        sd, kinds = {}, {}
        for t in GGUFReader(path).tensors:
            w = torch.from_numpy(dequantize(t.data, t.tensor_type).copy()).reshape([int(x) for x in reversed(t.shape)])
            part = t.name.split(".")
            if t.name == "token_embd.weight":
                k = "model.embed_tokens.weight"
            elif t.name == "output_norm.weight":
                k = "model.norm.weight"
            else:
                k = "model.layers." + part[1] + "." + NAMES[part[2]] + ".weight"
            sd[k] = w
            kinds[t.tensor_type.name] = kinds.get(t.tensor_type.name, 0) + 1
        return sd, kinds

    for q in ["f16", "Q8_0", "Q4_K_M", "Q3_K_M", "Q2_K"] if DEV.type == "cuda" else ["Q4_K_M"]:
        sd, kinds = gguf_weights(gguf[q]["file"])
        gap = max(float((sd[k] - merged_sd[k]).abs().max()) for k in merged_sd)
        load_into(sd)
        runs["gguf " + q] = panel(model)
        runs["gguf " + q]["kinds"], runs["gguf " + q]["MiB"] = kinds, gguf[q]["MiB"]
        show("GGUF " + q, runs["gguf " + q])
        print("   tensor types", kinds, "| largest weight change", round(gap, 4),
              "| Hindi bpb vs merged", ci(runs[best], runs["gguf " + q], "hi"), "| reply NLL vs merged", ci(runs[best], runs["gguf " + q], "reply"))
    load_into(merged_sd)

    # llama.cpp's own check: perplexity on the same Hindi texts, with llama.cpp's CPU kernels
    open("hindi.txt", "w", encoding="utf-8").write((NL + NL).join(hindi))
    for q in ["f16", "Q4_K_M", "Q2_K"]:
        sec, out = sh([BIN + "/llama-perplexity", "-m", gguf[q]["file"], "-f", "hindi.txt", "-c", "512", "--chunks", "40", "-t", str(os.cpu_count())])
        gguf[q]["ppl"] = [l for l in out.splitlines() if "Final estimate" in l][0].split("PPL = ")[1].strip()
        print(q.ljust(7), "llama-perplexity on the Hindi texts, 40 chunks of 512:", gguf[q]["ppl"], "|", sec, "s")
    runs["gguf"] = gguf

## Step 8. Quantize to 4 bits with GPTQ, with two calibration sets (GPU)

Code cell 8 on the lab page.

In [ ]:
import json, site, venv
from safetensors.torch import load_file

def side_env(path, packages, share=True):
    # A virtual environment for a tool that pins other versions. With share=True it also sees this runtime's
    # packages, torch included, but its own installs come first on its path.
    if not os.path.isdir(path):
        venv.create(path, with_pip=True)
        if share:
            purelib = sh([path + "/bin/python", "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"])[1].strip()
            open(purelib + "/runtime.pth", "w").write(NL.join(site.getsitepackages()))
        sh([path + "/bin/python", "-m", "pip", "install", "-q", *packages])

side_env("lc-env", ["llmcompressor==0.15.0"])          # llm-compressor 0.15.0 pins transformers 5.21.0

GPTQ = '''import json, sys, time, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from safetensors.torch import save_file
calib, out = sys.argv[1], sys.argv[2]
tok = AutoTokenizer.from_pretrained("merged")
model = AutoModelForCausalLM.from_pretrained("merged", dtype=torch.float16, device_map="cuda:0")
rows = json.load(open(calib))
ds = Dataset.from_dict(dict(tok(rows, max_length=512, truncation=True)))
t = time.time()
oneshot(model=model, dataset=ds, recipe=GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"]),
        max_seq_length=512, num_calibration_samples=len(rows))
sec = time.time() - t
model.save_pretrained(out, save_compressed=True)
tok.save_pretrained(out)
# Read the saved 4-bit checkpoint back, unpacked to fp16 weights, so the main notebook can score exactly what ships
from transformers import CompressedTensorsConfig
back = AutoModelForCausalLM.from_pretrained(out, dtype=torch.float16, quantization_config=CompressedTensorsConfig(run_compressed=False))
keep = {k: v.contiguous() for k, v in back.state_dict().items()
        if k != "lm_head.weight" and not k.endswith(("_scale", "_shape", "_zero_point", "_g_idx"))}
save_file(keep, out + ".fp16.safetensors")       # weights on the 4-bit grid, stored as fp16; scales dropped
print("RESULT", json.dumps({"oneshotSec": round(sec, 1)}))
'''
open("gptq.py", "w").write(GPTQ)

# Two calibration sets of 128 rows: English Wikipedia text, or text like what this model must keep doing
wiki = [r["text"] for r in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:3000]") if len(r["text"]) > 200][:128]
hi_train = [r["text"][:2000] for r in load_dataset("HuggingFaceFW/fineweb-2", name="hin_Deva", split="train", streaming=True).take(64)]
uf = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs").shuffle(seed=1).select(range(64))
own = hi_train + [TEMPLATE.format(ex["prompt"]) + ex["chosen"][-1]["content"].strip() + "<|endoftext|>" for ex in uf]
for name, rows in [("wikitext", wiki), ("own", own)]:
    json.dump(rows, open("calib-" + name + ".json", "w"))
    sec, out = sh(["lc-env/bin/python", "gptq.py", "calib-" + name + ".json", "gptq-" + name])
    size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk("gptq-" + name) for f in fs if f.endswith(".safetensors"))
    load_into(load_file("gptq-" + name + ".fp16.safetensors"))
    key = "gptq " + name
    runs[key] = panel(model)
    runs[key]["MiB"], runs[key]["sec"] = round(size / 2 ** 20, 1), json.loads(out.split("RESULT ")[1].splitlines()[0])["oneshotSec"]
    show("GPTQ W4A16, " + name, runs[key])
    print("   ", runs[key]["MiB"], "MiB of weights | GPTQ took", runs[key]["sec"], "s | Hindi bpb vs merged",
          ci(runs[best], runs[key], "hi"), "| reply NLL vs merged", ci(runs[best], runs[key], "reply"))
print("own minus wikitext calibration, Hindi bpb", ci(runs["gptq wikitext"], runs["gptq own"], "hi"),
      "| reply NLL", ci(runs["gptq wikitext"], runs["gptq own"], "reply"))
load_into(merged_sd)

## Step 9. When rounding fails, train with fake quantization (QAT, GPU; FROM_OUR_CHECKPOINT loads our result)

Code cell 9 on the lab page.

In [ ]:
import copy, random
import torch.nn.utils.parametrize as P

BITS, GROUP = 3, 32         # 3-bit weights in groups of 32 columns: round-to-nearest loses clearly at this size
RUN_QAT = DEV.type == "cuda" and not FROM_OUR_CHECKPOINT   # 150 training steps, 593 s on our A100 and 905 s on our T4; needs a GPU

def fake_quant(w, bits=BITS, group=GROUP):
    # Symmetric round-to-nearest per group of columns, returned in float: the values a 3-bit file would hold
    qmax = 2 ** (bits - 1) - 1
    g = w.reshape(-1, group)
    scale = (g.abs().amax(1, keepdim=True) / (qmax + 0.5)).clamp_min(torch.finfo(w.dtype).smallest_normal)
    return (torch.clamp(torch.round(g * (1.0 / scale)), -qmax - 1, qmax) * scale).reshape(w.shape)

class FakeQuant(torch.nn.Module):
    def forward(self, w):
        return w + (fake_quant(w) - w).detach()      # straight-through: forward sees the quantized weight, backward sees identity

def linears(m):
    return [mod for name, mod in m.named_modules() if isinstance(mod, torch.nn.Linear) and "lm_head" not in name]

def rtn(sd):
    # Post-training quantization: round every linear weight once, no training
    return {k: (fake_quant(v.float()) if v.dim() == 2 and "embed" not in k else v.float()) for k, v in sd.items()}

load_into(rtn(merged_sd))
runs["rtn 3-bit"] = panel(model)
show("RTN 3-bit, no training", runs["rtn 3-bit"])
load_into(merged_sd)

if RUN_QAT:
    # Distil the merged model into a copy that trains with 3-bit weights in the forward pass (LLM-QAT style)
    teacher = model
    student = copy.deepcopy(model).float().train()
    for mod in linears(student):
        P.register_parametrization(mod, "weight", FakeQuant())
    for name, p in student.named_parameters():
        p.requires_grad = "parametrizations" in name          # train the linear weights; embeddings and norms stay
    student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    student.config.use_cache = False
    hi_rows = [r["text"] for r in load_dataset("HuggingFaceFW/fineweb-2", name="hin_Deva", split="train", streaming=True).skip(64).take(600)]
    uf_rows = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs").shuffle(seed=2).select(range(600))
    texts = hi_rows + [TEMPLATE.format(e["prompt"]) + e["chosen"][-1]["content"].strip() + "<|endoftext|>" for e in uf_rows]
    random.Random(0).shuffle(texts)
    opt = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=1e-5, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", init_scale=2048)
    STEPS, MICRO, ACC = 150, 1, 8          # 8 sequences of up to 512 tokens per step
    t0, hist, i = time.time(), [], 0
    torch.cuda.reset_peak_memory_stats()
    for step in range(STEPS):
        opt.zero_grad(set_to_none=True)
        for _ in range(ACC):
            enc = tok(texts[i:i + MICRO], max_length=512, truncation=True, padding=True, return_tensors="pt").to(DEV)
            i = (i + MICRO) % (len(texts) - MICRO)
            with torch.no_grad():
                t_logp = torch.log_softmax(teacher(**enc).logits.float(), -1)
            with torch.autocast("cuda", dtype=torch.float16):
                s_logits = student(**enc).logits
            s_logp = torch.log_softmax(s_logits.float(), -1)
            kl = (t_logp.exp() * (t_logp - s_logp)).sum(-1)                  # forward KL per position
            loss = (kl * enc.attention_mask).sum() / enc.attention_mask.sum() / ACC
            scaler.scale(loss).backward()
            del t_logp, s_logp, s_logits
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for p in student.parameters() if p.requires_grad], 1.0)
        scaler.step(opt)
        scaler.update()
        hist.append(round(loss.item() * ACC, 4))
        if step % 25 == 0 or step == STEPS - 1:
            print("step", step, "| KL to the merged model", hist[-1], "|", round(time.time() - t0), "s")
    runs["qat"] = {"klFirst": hist[0], "klLast": hist[-1], "trainSec": round(time.time() - t0, 1),
                   "peakGB": round(torch.cuda.max_memory_allocated() / 1e9, 2), "hist": hist}
    for mod in linears(student):
        P.remove_parametrizations(mod, "weight", leave_parametrized=False)   # keep the trained float weights
    qat_sd = rtn({k: v for k, v in student.state_dict().items() if k != "lm_head.weight"})
    del student, opt
    gc.collect()
    torch.cuda.empty_cache()
    load_into(qat_sd)
    runs["qat 3-bit"] = panel(model)
    show("QAT 3-bit", runs["qat 3-bit"])
    print("   QAT minus RTN, Hindi bpb", ci(runs["rtn 3-bit"], runs["qat 3-bit"], "hi"), "| reply NLL",
          ci(runs["rtn 3-bit"], runs["qat 3-bit"], "reply"), "| train", runs["qat"]["trainSec"], "s, peak", runs["qat"]["peakGB"], "GB")
    load_into(merged_sd)
elif FROM_OUR_CHECKPOINT:
    q = REC["qat"]
    runs["qat"], runs["qat 3-bit"] = {k: q[k] for k in ("klFirst", "klLast", "trainSec", "peakGB")}, q["qat"]
    print("QAT, recorded on our A100 (set FROM_OUR_CHECKPOINT = False to train it yourself)")
    print("   KL to the merge", q["klFirst"], "at step 0 and", q["klLast"], "at step", q["steps"] - 1, "| trained", q["trainSec"], "s, peak", q["peakGB"], "GB")
    for name, m in (("RTN 3-bit", q["rtn"]), ("QAT 3-bit", q["qat"])):
        print("   " + name.ljust(10), "| Hindi bpb", format(m["hiBpb"], ".4f"), "| English bpb", format(m["enBpb"], ".4f"),
              "| reply NLL", format(m["replyNll"], ".4f"), "| ends", format(m["endRate"], ".2f"))

## Step 10. Export with the right config, chat template and model card

Code cell 10 on the lab page.

In [ ]:
from transformers import GenerationConfig
PUSH_TO = None         # your own Hub repo, for example "your-name/qwen3-0.6b-hindi-instruct-merge"; needs an HF_TOKEN with write access

# 1. Generation settings that match how the model was trained: stop at <|endoftext|>, no 2,048-token default
GenerationConfig(eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id, max_new_tokens=256, do_sample=False).save_pretrained("merged")

# 2. Prove the saved chat template writes exactly the prompt format the SFT model learned
reloaded = AutoTokenizer.from_pretrained("merged")
chat = [{"role": "user", "content": "Name a fruit."}]
print("template matches training format:", reloaded.apply_chat_template(chat, tokenize=False, add_generation_prompt=True) == TEMPLATE.format("Name a fruit."))

# 3. The model card: what it is, what it was trained on, what we measured, and where it fails
m, b = runs[best], runs["base"]
CARD = f"""---
license: apache-2.0
base_model: Qwen/Qwen3-0.6B-Base
tags: [merge, task-arithmetic, llm-training-lab]
datasets: [HuggingFaceH4/ultrafeedback_binarized, HuggingFaceFW/fineweb-2]
language: [en, hi]
---
# Qwen3-0.6B-Base, Hindi continued pretraining merged with instruction SFT

A weight merge ({best}) of two fine-tunes of Qwen/Qwen3-0.6B-Base, built in the LLM Training Lab:
an instruction SFT (LoRA, merged, 3,000 UltraFeedback chosen replies) and Hindi continued pretraining (320 steps, 5% English replay).

## Prompt format
Plain text, not ChatML. The base model's chat tokens were never trained, so the chat template writes:
"### Instruction:", the request, a blank line, "### Response:", and the reply ends with <|endoftext|>.

## Evaluation (held-out, paired against the base model)
| | Base | This model |
|---|---|---|
| Hindi bits per byte, 200 FineWeb-2 test texts | {b['hiBpb']:.4f} | {m['hiBpb']:.4f} |
| English bits per byte, 200 FineWeb-Edu texts | {b['enBpb']:.4f} | {m['enBpb']:.4f} |
| Reply NLL per token, 300 UltraFeedback test replies | {b['replyNll']:.4f} | {m['replyNll']:.4f} |
| Replies that stop on their own, 100 prompts, 256 tokens | {b.get('endRate', float('nan')):.2f} | {m.get('endRate', float('nan')):.2f} |

## Limits
- 0.6B parameters: replies are often wrong or repetitive. Not for factual, medical, legal or safety-relevant use.
- No safety tuning and no refusal training, and we did not measure refusals. Do not deploy it to the public.
- The Hindi gain is a small drop in bits per byte, not Hindi instruction following, which was never trained or measured.
- UltraFeedback labels come from GPT-4; OpenAI's terms on model outputs may apply to what you build with them.

## Licence
Apache-2.0, as the base model. UltraFeedback binarized is MIT; FineWeb-2 is ODC-By 1.0.
"""
open("merged/README.md", "w", encoding="utf-8").write(CARD)
print(sorted(os.listdir("merged")))

if PUSH_TO:
    from huggingface_hub import HfApi
    HfApi().create_repo(PUSH_TO, exist_ok=True)
    print(HfApi().upload_folder(folder_path="merged", repo_id=PUSH_TO))

## Step 11. Serve with llama.cpp and run a smoke eval through the endpoint

Code cell 11 on the lab page.

In [ ]:
import urllib.request

def post(url, body):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=600).read())

def serve(gguf_file, port, extra=()):
    # llama-server on the CPU, OpenAI-compatible, using the chat template stored in the GGUF file (--jinja)
    p = subprocess.Popen([BIN + "/llama-server", "-m", gguf_file, "--port", str(port), "-c", "2048", "--jinja",
                          "-t", str(os.cpu_count()), *extra], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(120):
        try:
            if json.loads(urllib.request.urlopen("http://127.0.0.1:" + str(port) + "/health").read())["status"] == "ok":
                return p
        except Exception:
            time.sleep(1)
    p.kill()
    raise RuntimeError("llama-server did not start")

def smoke(port, n=20):
    # The post-deployment smoke eval: same prompts as the panel, through the endpoint a user would call
    url = "http://127.0.0.1:" + str(port)
    stops, leaks, same_ids, outs = 0, 0, 0, []
    t = time.time()
    for q in prompts[:n]:
        r = post(url + "/v1/chat/completions", {"messages": [{"role": "user", "content": q}], "max_tokens": 256, "temperature": 0})
        text = r["choices"][0]["message"]["content"]
        stops += r["choices"][0]["finish_reason"] == "stop"
        leaks += "### Instruction" in text or "<|im_start|>" in text
        same_ids += post(url + "/tokenize", {"content": TEMPLATE.format(q)})["tokens"] == tok(TEMPLATE.format(q)).input_ids
        outs.append(text)
    return {"stopRate": stops / n, "leakRate": leaks / n, "sameTokenIds": same_ids / n, "sec": round(time.time() - t, 1), "first": outs[0]}

server = serve("merged-Q4_K_M.gguf", 8090)
runs["serve ours"] = smoke(8090)
server.kill()
print("our template    ", {k: v for k, v in runs["serve ours"].items() if k != "first"})
print("   first reply:", runs["serve ours"]["first"][:300])

# The trap: serve the same file with ChatML, the usual Qwen chat format, whose control tokens this base model never trained
server = serve("merged-Q4_K_M.gguf", 8091, ["--chat-template", "chatml"])
runs["serve chatml"] = smoke(8091)
server.kill()
print("ChatML template ", {k: v for k, v in runs["serve chatml"].items() if k != "first"})
print("   first reply:", runs["serve chatml"]["first"][:300])

# For Ollama on your own machine: the same GGUF file, with the template written out in Ollama's own format
MODELFILE = '''FROM ./merged-Q4_K_M.gguf
TEMPLATE """### Instruction:
{{ .Prompt }}

### Response:
"""
PARAMETER stop "<|endoftext|>"
PARAMETER temperature 0
'''
open("Modelfile", "w").write(MODELFILE)
json.dump(prompts[:20], open("prompts.json", "w"))         # the same 20 prompts, for the Ollama check

## Step 12. Serve with vLLM, one merged model or one base with adapters (RUN_VLLM = True; FROM_OUR_CHECKPOINT prints our run)

Code cell 12 on the lab page.

In [ ]:
RUN_VLLM = False      # True builds a separate vLLM environment (it pins its own torch) and serves on this GPU; recorded on an A100.
                      # With FROM_OUR_CHECKPOINT = True and this False, step 12 prints our recorded vLLM numbers

# 1. Take the SFT fine-tune apart again: its task vector has rank 16 per layer, so an SVD recovers a LoRA adapter
from safetensors.torch import save_file
lora, worst = {}, 0.0
for k in W["base"]:
    if k.endswith("proj.weight") and "embed" not in k:
        d = tau("sft", k)
        U, S, Vh = torch.linalg.svd(d, full_matrices=False)
        B, A = U[:, :16] * S[:16].sqrt(), S[:16].sqrt()[:, None] * Vh[:16]
        worst = max(worst, float((B @ A - d).norm() / d.norm()))
        name = "base_model.model." + k[:-len(".weight")]
        lora[name + ".lora_A.weight"], lora[name + ".lora_B.weight"] = A.contiguous(), B.contiguous()
os.makedirs("sft-lora", exist_ok=True)
save_file(lora, "sft-lora/adapter_model.safetensors")
json.dump({"base_model_name_or_path": BASE, "peft_type": "LORA", "task_type": "CAUSAL_LM", "r": 16, "lora_alpha": 16,
           "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]},
          open("sft-lora/adapter_config.json", "w"))
print(len(lora) // 2, "layers | largest relative error of the rank-16 rebuild", round(worst, 4))

# 2. One merged model, or one base model with several adapters swapped per request
VLLM = '''import json, sys, time
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
mode, prompts = sys.argv[1], json.load(open("vllm_prompts.json"))
sp = SamplingParams(temperature=0, max_tokens=256)
if mode == "multi-lora":
    llm = LLM(model=sys.argv[2], dtype="float16", max_model_len=1024, gpu_memory_utilization=0.5, enable_lora=True, max_lora_rank=16, max_loras=2)
    reqs = [LoRARequest("sft", 1, "sft-lora"), LoRARequest("math", 2, sys.argv[3])]
    t = time.time()
    outs = llm.generate(prompts + prompts, sp, lora_request=[reqs[0]] * len(prompts) + [reqs[1]] * len(prompts))
else:
    llm = LLM(model=sys.argv[2], dtype="float16", max_model_len=1024, gpu_memory_utilization=0.5)
    t = time.time()
    outs = llm.generate(prompts, sp)
sec = time.time() - t
toks = [list(o.outputs[0].token_ids) for o in outs]
print("RESULT", json.dumps({"sec": round(sec, 2), "tokens": sum(map(len, toks)), "requests": len(toks),
                            "stopRate": sum(o.outputs[0].finish_reason == "stop" for o in outs) / len(outs), "ids": toks}))
'''
if RUN_VLLM:
    side_env("vllm-env", ["vllm==0.33.0"], share=False)      # vLLM 0.33.0 brings its own torch 2.14.0
    open("vllm_job.py", "w").write(VLLM)
    json.dump([TEMPLATE.format(q) for q in prompts], open("vllm_prompts.json", "w"))
    math_lora = snapshot_download(HUB, allow_patterns=["lltDistillation/onpolicy-reverse-s0/*"]) + "/lltDistillation/onpolicy-reverse-s0"
    model.to("cpu")                                        # give the GPU to vLLM
    torch.cuda.empty_cache()
    res = {}
    for mode, args in [("merged", ["merged"]), ("sft-full", [DIRS["sft"]]), ("multi-lora", [DIRS["base"], math_lora])]:
        # The FlashInfer sampler compiles a CUDA kernel on first use and needs ninja and nvcc on PATH; vLLM's PyTorch sampler needs neither
        sec, out = sh(["vllm-env/bin/python", "vllm_job.py", mode, *args], env=dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0"))
        res[mode] = json.loads(out.split("RESULT ")[1].splitlines()[0])
        r = res[mode]
        print(mode.ljust(10), r["requests"], "requests |", r["tokens"], "tokens in", r["sec"], "s =", round(r["tokens"] / r["sec"]),
              "tokens/s | stop rate", round(r["stopRate"], 2), "| process", sec, "s")
    n = len(prompts)
    same = sum(a == b for a, b in zip(res["sft-full"]["ids"], res["multi-lora"]["ids"][:n]))
    print("SFT model vs base + rebuilt SFT adapter: identical greedy replies", same, "of", n)
    runs["vllm"] = {k: {kk: vv for kk, vv in v.items() if kk != "ids"} for k, v in res.items()}
    runs["vllm"]["identical"] = same
    model.to(DEV)
elif FROM_OUR_CHECKPOINT:
    runs["vllm"] = REC["vllm"]
    for mode in ("merged", "sft-full", "multi-lora"):
        r = runs["vllm"][mode]
        print(mode.ljust(10), r["requests"], "requests |", r["tokens"], "tokens in", r["sec"], "s =", round(r["tokens"] / r["sec"]),
              "tokens/s | stop rate", round(r["stopRate"], 2), "| recorded on our A100")
    print("SFT model vs base + rebuilt SFT adapter: identical greedy replies", runs["vllm"]["identical"], "of", runs["vllm"]["merged"]["requests"], "| recorded")